In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/cleaned_delivery.csv")

df["Order_Date"] = pd.to_datetime(df["Order_Date"])

df["Time_Orderd"] = pd.to_datetime(
    df["Time_Orderd"],
    errors="coerce"
)

# Calendar Features
df["Year"] = df["Order_Date"].dt.year
df["Month"] = df["Order_Date"].dt.month
df["Month_Name"] = df["Order_Date"].dt.month_name()
df["Weekday"] = df["Order_Date"].dt.day_name()
df["Quarter"] = df["Order_Date"].dt.quarter

# Time Features
df["Order_Hour"] = df["Time_Orderd"].dt.hour

df["Weekend"] = (
    df["Weekday"]
    .isin(["Saturday","Sunday"])
    .astype(int)
)

df["Peak_Hour"] = np.where(
    (
        (df["Order_Hour"]>=11)
        &
        (df["Order_Hour"]<=14)
    )
    |
    (
        (df["Order_Hour"]>=18)
        &
        (df["Order_Hour"]<=22)
    ),
    "Peak",
    "Non Peak"
)

# Distance
df["Distance"] = np.sqrt(
    (df["Restaurant_latitude"]-
    df["Delivery_location_latitude"])**2
    +
    (df["Restaurant_longitude"]-
    df["Delivery_location_longitude"])**2
)

# Delivery Speed
df["Delivery_Speed"] = (
    df["Distance"]/
    df["Time_taken(min)"]
)

# Categories
df["Late_Delivery"] = (
    df["Time_taken(min)"]>40
).astype(int)

df["High_Traffic"] = (
    df["Road_traffic_density"]
    .isin(["Jam","High"])
    .astype(int)
)

df["Festival_Order"] = (
    df["Festival"]=="Yes"
).astype(int)

vehicle_score = {
    "motorcycle":4,
    "electric_scooter":5,
    "scooter":3,
    "bicycle":2
}

df["Vehicle_Score"] = (
    df["Type_of_vehicle"]
    .str.lower()
    .map(vehicle_score)
)

df["Driver_Efficiency"] = (
    df["Delivery_person_Ratings"]*10
)/df["Time_taken(min)"]

df.to_csv(
    "../data/processed/delivery_features.csv",
    index=False
)

print(df.head())

print("Feature Engineering Completed")